In [2]:
import pandas as pd
df = pd.read_csv("cleaned_data.csv")
df.head()

,order_id,customer_age,purchase_amount,quantity,discount_percent,customer_rating,delivery_days,product_category,payment_method,satisfaction_score,is_returned
0,1,40.0,58.15,4,5.0,5.0,11.0,Clothing,Debit Card,99.4,1
1,2,34.0,24.30,4,5.0,NaN,7.0,Clothing,UPI,99.9,0
2,3,41.0,35.08,1,10.0,4.8,7.0,Clothing,Credit Card,99.7,0
3,4,50.0,25.09,5,25.0,NaN,4.0,Clothing,Debit Card,99.9,0
4,5,33.0,25.52,7,0.0,3.1,5.0,Electronics,Cash on Delivery,92.0,0


In [ ]:
print(df.columns.tolist())
print(df.dtypes)

In [ ]:
TARGET_COLUMN = "purchase_amount"
print(df["product_category"].unique())
print(df["payment_method"].unique())

ORDINAL_COLUMNS = []
RANDOM_STATE = 42
print(TARGET_COLUMN in df.columns)   # should print True
print(all(col in df.columns for col in ORDINAL_COLUMNS))  # True (trivially, since list is empty)

In [ ]:
"""
Part 2 — Supervised ML Model: Build, Train, Evaluate
Run: python part2_supervised_ml.py
Requires: pandas, numpy, scikit-learn, imbalanced-learn, matplotlib
"""

# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)

# CONFIG — your dataset's actual values

TARGET_COLUMN = "purchase_amount"
ORDINAL_COLUMNS = []
RANDOM_STATE = 42

# Task 1 — Load data, define X, y_reg, y_clf

df = pd.read_csv("cleaned_data.csv")

y_reg = df[TARGET_COLUMN].copy()

# Drop target AND is_returned (is_returned becomes y_clf, not a feature)
X = df.drop(columns=[TARGET_COLUMN, "is_returned"]).copy()

# Fill missing values (customer_rating had 216 NaNs)
X = X.fillna(X.median(numeric_only=True))

# Natural binary label from the dataset
y_clf = df["is_returned"].copy()

print("y_reg target:", TARGET_COLUMN)
print("y_clf label: is_returned (natural binary column, 1 = returned, 0 = not returned)")
print("\ny_clf class balance:\n", y_clf.value_counts(normalize=True))
print("\nMissing values in X after fill:\n", X.isnull().sum().sum())
print("\nX columns:\n", X.columns.tolist())


In [ ]:
# Task 2 — Encode categorical columns

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
nominal_cols = [c for c in categorical_cols if c not in ORDINAL_COLUMNS]

# Ordinal encoding (label encoding preserving order) — you must define the order per column if needed
for col in ORDINAL_COLUMNS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Nominal encoding (one-hot, drop first to avoid multicollinearity)
if nominal_cols:
    X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

print("\nFinal feature columns after encoding:\n", X.columns.tolist())


In [ ]:
# Task 3 — Leak-free train-test split and scaling

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
scaler.fit(X_train)                      # fit ONLY on training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Task 4 — Regression: Linear Regression + Ridge

lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_reg_train)
y_pred_reg = lin_reg.predict(X_test_scaled)

mse_lin = mean_squared_error(y_reg_test, y_pred_reg)
r2_lin = r2_score(y_reg_test, y_pred_reg)

print("\n--- Linear Regression ---")
print("MSE:", mse_lin, "| R2:", r2_lin)

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lin_reg.coef_
}).sort_values(by="coefficient", key=abs, ascending=False)
print("\nLinear Regression coefficients (sorted by magnitude):\n", coef_table)
print("\nTop 3 features by |coefficient|:\n", coef_table.head(3))

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

comparison_table = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge (alpha=1.0)"],
    "MSE": [mse_lin, mse_ridge],
    "R2": [r2_lin, r2_ridge]
})
print("\n--- Linear vs Ridge comparison ---\n", comparison_table)


In [ ]:
# Task 5 — Classification: Logistic Regression

print("\n--- Class balance check (train) ---")
print(y_clf_train.value_counts(normalize=True))

# Imbalance handling: use class_weight='balanced' (documented choice)
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train_scaled, y_clf_train)

y_pred_clf = log_reg.predict(X_test_scaled)
y_proba_clf = log_reg.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_clf_test, y_pred_clf)
report = classification_report(y_clf_test, y_pred_clf)
acc = accuracy_score(y_clf_test, y_pred_clf)

print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)
print("Accuracy:", acc)

fpr, tpr, thresholds = roc_curve(y_clf_test, y_proba_clf)
auc_baseline = roc_auc_score(y_clf_test, y_proba_clf)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_baseline:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Logistic Regression (C=1.0)")
plt.legend()
plt.savefig("roc_curve.png")
plt.close()
print("\nAUC:", auc_baseline, "(saved plot as roc_curve.png)")

# Task 5b — Decision-threshold sensitivity

print("\n--- Threshold sensitivity ---")
rows = []
for t in np.arange(0.30, 0.71, 0.10):
    preds_t = (y_proba_clf >= t).astype(int)
    p = precision_score(y_clf_test, preds_t, zero_division=0)
    r = recall_score(y_clf_test, preds_t, zero_division=0)
    f1 = f1_score(y_clf_test, preds_t, zero_division=0)
    rows.append({"Threshold": round(t, 2), "Precision": p, "Recall": r, "F1": f1})

threshold_table = pd.DataFrame(rows)
print(threshold_table)
best_row = threshold_table.loc[threshold_table["F1"].idxmax()]
print("\nThreshold maximizing F1:", best_row["Threshold"])


In [ ]:
# Task 6 — Regularization experiment

log_reg_strong = LogisticRegression(max_iter=1000, class_weight="balanced", C=0.01)
log_reg_strong.fit(X_train_scaled, y_clf_train)
y_proba_strong = log_reg_strong.predict_proba(X_test_scaled)[:, 1]
y_pred_strong = log_reg_strong.predict(X_test_scaled)

precision_strong = precision_score(y_clf_test, y_pred_strong, zero_division=0)
recall_strong = recall_score(y_clf_test, y_pred_strong, zero_division=0)
auc_strong = roc_auc_score(y_clf_test, y_proba_strong)

precision_base = precision_score(y_clf_test, y_pred_clf, zero_division=0)
recall_base = recall_score(y_clf_test, y_pred_clf, zero_division=0)

reg_comparison = pd.DataFrame({
    "Model": ["C=1.0 (baseline)", "C=0.01 (strong L2)"],
    "Precision": [precision_base, precision_strong],
    "Recall": [recall_base, recall_strong],
    "AUC": [auc_baseline, auc_strong]
})
print("\n--- Regularization comparison ---\n", reg_comparison)


In [ ]:
# Task 7 — Bootstrap confidence interval for AUC difference

n_boot = 500
auc_diffs = []
y_clf_test_arr = np.array(y_clf_test)

for _ in range(n_boot):
    idx = np.random.choice(len(y_clf_test_arr), size=len(y_clf_test_arr), replace=True)
    y_sample = y_clf_test_arr[idx]

    # skip degenerate samples with only one class present
    if len(np.unique(y_sample)) < 2:
        continue

    auc_c1 = roc_auc_score(y_sample, y_proba_clf[idx])
    auc_c001 = roc_auc_score(y_sample, y_proba_strong[idx])
    auc_diffs.append(auc_c1 - auc_c001)

auc_diffs = np.array(auc_diffs)
mean_diff = auc_diffs.mean()
ci_lower = np.percentile(auc_diffs, 2.5)
ci_upper = np.percentile(auc_diffs, 97.5)

print("\n--- Bootstrap AUC difference (C=1.0 minus C=0.01) ---")
print("Mean AUC difference:", mean_diff)
print("95% CI:", (ci_lower, ci_upper))
print("Interval excludes zero:", not (ci_lower <= 0 <= ci_upper))